# ViHSD · PhoBERT baseline đầu tiên

**Mục tiêu:** kiểm tra dữ liệu → EDA → RDRSegmenter → PhoBERT → kiểm tra học → fine-tune → đánh giá test gốc.

- Baseline mới phục vụ robustness; không gọi là tái lập một điểm PhoBERT trong paper ViHSD 2021.
- Nguồn dữ liệu: archive tại repo chính thức của tác giả, pin commit và SHA-256 trong code.
- Giữ nguyên split; 2 dòng train rỗng được loại với ledger cụ thể. Validation/test không thay đổi.
- Giữ emoji, case, dấu, stopword, teencode và ký tự kéo dài.
- Chọn model bằng **validation Macro-F1**. Test gốc chỉ đánh giá sau khi chọn checkpoint.
- Một GPU, seed 42, 5 epoch. Không thêm nhiễu hoặc FGM trong lần này.

Artifact ở `/kaggle/working/vihsd_project/`: `reports/`, `runs/`, dữ liệu và config để tái hiện.
Source được nhúng trực tiếp từ repo TMG để notebook không phụ thuộc một GitHub cá nhân chưa publish.


## 1. Môi trường và source snapshot

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["MPLBACKEND"] = "Agg"
WORK = Path("/kaggle/working")
ROOT = WORK / "vihsd_project"
ROOT.mkdir(parents=True, exist_ok=True)
# Retain the CUDA-compatible PyTorch already installed by Kaggle.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "transformers==4.57.1", "huggingface-hub>=0.34,<1", "accelerate>=1.2,<2", "py-vncorenlp==0.1.4",
    "seaborn>=0.13,<0.14", "pyarrow>=17", "scikit-learn>=1.5", "pandas>=2.2,<4"])
if not shutil.which("java"):
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call(["apt-get", "install", "-y", "-qq", "default-jre-headless"])
java_home = str(Path(shutil.which("java")).resolve().parent.parent)
os.environ["JAVA_HOME"] = java_home
os.environ["JDK_HOME"] = java_home
print(subprocess.check_output(["nvidia-smi"], text=True))
import torch
assert torch.cuda.is_available(), "Full baseline requires a GPU. Enable GPU in Kaggle settings."
assert torch.cuda.device_count() == 1, "This protocol uses exactly one visible GPU"
print("GPU:", torch.cuda.get_device_name(0), "Python:", sys.version)


In [ ]:
SOURCE_SNAPSHOT = {'vihsd/__init__.py': '"""Reproducible ViHSD data, EDA and PhoBERT baseline."""\n', 'vihsd/cli.py': "import argparse\nimport json\nfrom pathlib import Path\nfrom .data import load_raw, audit, prepare\nfrom .eda import create_eda\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument('command', choices=['doctor', 'download', 'audit', 'eda', 'prepare', 'train', 'evaluate'])\n    parser.add_argument('--root', default='.')\n    parser.add_argument('--config', default='configs/baseline.json')\n    parser.add_argument('--run-dir')\n    args = parser.parse_args()\n    root = Path(args.root).resolve()\n    if args.command == 'doctor':\n        from .pipeline import ensure_java\n        from .training import environment\n        print(ensure_java()); print(json.dumps(environment(), indent=2)); return\n    if args.command == 'download':\n        frames = load_raw(root); print({s: len(f) for s, f in frames.items()}); return\n    if args.command in ['audit', 'eda']:\n        frames = load_raw(root)\n        summary = audit(frames, root / 'reports/audit')\n        if args.command == 'eda':\n            create_eda(frames, root / 'reports/eda', summary)\n        print(json.dumps(summary, ensure_ascii=False, indent=2)); return\n    config = json.loads(Path(args.config).read_text())\n    from .pipeline import prepare_all, run_training\n    if args.command == 'prepare':\n        prepare_all(root, config)\n    elif args.command == 'train':\n        run_training(root, config)\n    else:\n        if not args.run_dir:\n            parser.error('--run-dir is required for evaluate')\n        from transformers import AutoTokenizer\n        from .training import tokenize_frames, evaluate_checkpoint\n        run_dir = Path(args.run_dir)\n        saved_config = json.loads((run_dir / 'config.json').read_text())\n        tokenizer = AutoTokenizer.from_pretrained(run_dir / 'best', use_fast=False)\n        from .pipeline import ensure_java\n        ensure_java()\n        frames = prepare(root)\n        encoded, _ = tokenize_frames(frames, tokenizer, saved_config['max_length'], root / 'data/tokenized')\n        evaluate_checkpoint(run_dir / 'best', frames, encoded, tokenizer, run_dir / 'evaluation', saved_config['eval_batch_size'])\n\nif __name__ == '__main__':\n    main()\n", 'vihsd/data.py': '"""Pinned author data, immutable raw text and explicit split audits."""\nfrom __future__ import annotations\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\n\nDATA_COMMIT = \'70fe50c9c94145e3b9be5614a048cbd5b480fa54\'\nDATA_URL = f\'https://raw.githubusercontent.com/sonlam1102/vihsd/{DATA_COMMIT}/data/vihsd.zip\'\n# Official author repository blob fb17d90cbcb731b173f4f76977b3dfc78420ac7d.\nEXPECTED_SHA256 = \'1d823c4c86a8aede83555c64da4e1be24146967aac6b30cdaf8603d4910f328d\'\nSPLITS = {\'train\': \'train\', \'validation\': \'dev\', \'test\': \'test\'}\nEXPECTED_COUNTS = {\'train\': 24048, \'validation\': 2672, \'test\': 6680}\nLABELS = [\'CLEAN\', \'OFFENSIVE\', \'HATE\']\nPREPROCESS_VERSION = \'nfc-space-rdr-utf8-v2\'\nKNOWN_EMPTY_TRAIN_IDS = {\'train:010950\', \'train:020880\'}\n\n\ndef dump_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\n\n\ndef sha256_bytes(content: bytes) -> str:\n    return hashlib.sha256(content).hexdigest()\n\n\ndef normalize_text(text: str) -> str:\n    if not isinstance(text, str):\n        raise ValueError(\'Text must be a string; null must not become "nan"\')\n    text = unicodedata.normalize(\'NFC\', text)\n    # Preserve emoji joiners/variation selectors; remove only actual control chars.\n    text = \'\'.join(\' \' if unicodedata.category(c) == \'Cc\' else c for c in text)\n    return re.sub(r\'\\s+\', \' \', text).strip()\n\n\ndef download(root: str | Path) -> Path:\n    raw = Path(root) / \'data/raw\'\n    raw.mkdir(parents=True, exist_ok=True)\n    archive = raw / \'vihsd.zip\'\n    if not archive.exists():\n        tmp = archive.with_suffix(\'.tmp\')\n        with urllib.request.urlopen(DATA_URL, timeout=120) as response:\n            tmp.write_bytes(response.read())\n        tmp.replace(archive)\n    content = archive.read_bytes()\n    checksum = sha256_bytes(content)\n    if EXPECTED_SHA256 != \'TO_BE_PINNED\' and checksum != EXPECTED_SHA256:\n        raise ValueError(f\'Dataset checksum mismatch: {checksum}\')\n    manifest = {\'source_url\': DATA_URL, \'source_commit\': DATA_COMMIT,\n                \'sha256\': checksum, \'observed_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'labels\': dict(enumerate(LABELS)), \'files\': {}}\n    with zipfile.ZipFile(io.BytesIO(content)) as z:\n        for split, source_name in SPLITS.items():\n            member = f\'vihsd/{source_name}.csv\'\n            data = z.read(member)\n            target = raw / f\'{split}.csv\'\n            if target.exists() and target.read_bytes() != data:\n                raise ValueError(f\'Refusing to overwrite modified raw file: {target}\')\n            target.write_bytes(data)\n            manifest[\'files\'][split] = {\'member\': member, \'sha256\': sha256_bytes(data)}\n    dump_json(raw / \'manifest.json\', manifest)\n    return raw\n\n\ndef canonicalize(frame: pd.DataFrame, split: str) -> pd.DataFrame:\n    required = {\'free_text\', \'label_id\'}\n    if not required.issubset(frame.columns):\n        raise ValueError(f\'{split}: expected columns {sorted(required)}\')\n    out = frame[[\'free_text\', \'label_id\']].rename(columns={\'free_text\': \'text_raw\', \'label_id\': \'label\'}).copy()\n    out.insert(0, \'source_row\', range(len(out)))\n    out.insert(0, \'split\', split)\n    out.insert(0, \'sample_id\', [f\'{split}:{i:06d}\' for i in range(len(out))])\n    return out\n\n\ndef load_raw(root: str | Path) -> dict[str, pd.DataFrame]:\n    raw = download(root)\n    return {split: canonicalize(pd.read_csv(raw / f\'{split}.csv\', keep_default_na=False), split)\n            for split in SPLITS}\n\n\ndef audit(frames: dict[str, pd.DataFrame], output: str | Path, enforce_counts=True) -> dict:\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    errors = []\n    exclusions = []\n    summaries = {}\n    all_rows = []\n    for split, frame in frames.items():\n        valid_label = frame.label.isin([0, 1, 2])\n        valid_text = frame.text_raw.map(lambda x: isinstance(x, str) and bool(normalize_text(x)))\n        for row in frame.loc[~valid_label | ~valid_text].itertuples():\n            issue = {\'sample_id\': row.sample_id, \'invalid_label\': not bool(valid_label.loc[row.Index]), \'invalid_text\': not bool(valid_text.loc[row.Index])}\n            if split == \'train\' and row.sample_id in KNOWN_EMPTY_TRAIN_IDS and issue[\'invalid_text\'] and not issue[\'invalid_label\']:\n                exclusions.append({**issue, \'reason\': \'Empty author-provided train text; exclude from training only\'})\n            else:\n                errors.append(issue)\n        summaries[split] = {\'n\': len(frame), \'label_counts\': {str(k): int((frame.label == k).sum()) for k in range(3)},\n                            \'invalid_label\': int((~valid_label).sum()), \'invalid_text\': int((~valid_text).sum()),\n                            \'replacement_char_rows\': int(frame.text_raw.map(lambda x: \'\\ufffd\' in str(x)).sum()),\n                            \'raw_underscore_rows\': int(frame.text_raw.map(lambda x: \'_\' in str(x)).sum())}\n        if enforce_counts and len(frame) != EXPECTED_COUNTS[split]:\n            errors.append({\'split\': split, \'reason\': f\'Expected {EXPECTED_COUNTS[split]}, got {len(frame)}\'})\n        f = frame.copy()\n        f[\'raw_hash\'] = f.text_raw.map(lambda x: sha256_bytes(str(x).encode()))\n        f[\'normalized_hash\'] = f.text_raw.map(lambda x: sha256_bytes(normalize_text(x).encode()) if isinstance(x, str) else \'\')\n        all_rows.append(f)\n    combined = pd.concat(all_rows, ignore_index=True)\n    duplicate_stats = {}\n    for mode in [\'raw\', \'normalized\']:\n        key = mode + \'_hash\'\n        duplicates = combined[combined.duplicated(key, keep=False)].copy()\n        duplicates.to_csv(output / f\'duplicates_{mode}.csv\', index=False)\n        groups = combined.groupby(key)\n        cross_hashes = groups.split.nunique()\n        conflicts = groups.label.nunique()\n        cross = combined[combined[key].isin(cross_hashes[cross_hashes > 1].index)]\n        cross.to_csv(output / f\'cross_split_{mode}.csv\', index=False)\n        combined[combined[key].isin(conflicts[conflicts > 1].index)].to_csv(output / f\'label_conflicts_{mode}.csv\', index=False)\n        duplicate_stats[mode] = {\'duplicate_rows\': len(duplicates), \'duplicate_groups\': int(duplicates[key].nunique()),\n                                 \'cross_split_groups\': int((cross_hashes > 1).sum()),\n                                 \'cross_split_rows\': len(cross), \'conflicting_label_groups\': int((conflicts > 1).sum())}\n    summary = {\'splits\': summaries, \'duplicates\': duplicate_stats, \'blocking_errors\': errors, \'training_exclusions\': exclusions,\n               \'policy\': \'Preserve official rows/splits; report duplicates and conflicts. Never silently drop rows.\'}\n    dump_json(output / \'audit.json\', summary)\n    dump_json(output / \'training_exclusions.json\', exclusions)\n    if errors:\n        dump_json(output / \'blocking_errors.json\', errors)\n        raise ValueError(f\'Data audit failed; see {output}/blocking_errors.json\')\n    return summary\n\n\ndef preserve_java_unicode(segmenter):\n    """Avoid pyjnius String(str) truncating supplementary Unicode to 16 bits.\n\n    Java String(byte[], UTF-8) roundtrips emoji and other non-BMP characters.\n    Patching this constructor preserves VnCoreNLP\'s official word_segment logic.\n    """\n    java_string = segmenter.javaclass_String\n    segmenter.javaclass_String = lambda text: java_string(list(text.encode(\'utf-8\')), \'UTF-8\')\n    return segmenter\n\n\ndef supplementary_sequence(text):\n    return [c for c in text if ord(c) > 0xFFFF]\n\n\ndef prepare(root: str | Path, segmenter_dir: str | Path | None = None) -> dict[str, pd.DataFrame]:\n    """One sequential JVM, cached by full data/model/preprocessing fingerprint."""\n    import importlib.metadata\n    import py_vncorenlp\n    from tqdm.auto import tqdm\n    root = Path(root).resolve()\n    frames = load_raw(root)\n    audit(frames, root / \'reports/audit\')\n    frames[\'train\'] = frames[\'train\'].loc[~frames[\'train\'].sample_id.isin(KNOWN_EMPTY_TRAIN_IDS)].copy().reset_index(drop=True)\n    model_dir = Path(segmenter_dir or root / \'.cache/vncorenlp\').resolve()\n    model_dir.mkdir(parents=True, exist_ok=True)\n    if not (model_dir / \'VnCoreNLP-1.2.jar\').exists():\n        py_vncorenlp.download_model(save_dir=str(model_dir))\n    resources = sorted(p for p in model_dir.rglob(\'*\') if p.is_file())\n    resource_hashes = {str(p.relative_to(model_dir)): sha256_bytes(p.read_bytes()) for p in resources}\n    signature = {\'data_sha256\': sha256_bytes((root / \'data/raw/vihsd.zip\').read_bytes()),\n                 \'preprocessing\': PREPROCESS_VERSION, \'py_vncorenlp\': importlib.metadata.version(\'py-vncorenlp\'),\n                 \'resource_hashes\': resource_hashes}\n    signature_hash = sha256_bytes(json.dumps(signature, sort_keys=True).encode())\n    target = root / \'data/processed\' / signature_hash[:16]\n    metadata = target / \'manifest.json\'\n    if metadata.exists() and all((target / f\'{s}.parquet\').exists() for s in SPLITS):\n        if json.loads(metadata.read_text()) == signature:\n            return {s: pd.read_parquet(target / f\'{s}.parquet\') for s in SPLITS}\n    target.mkdir(parents=True, exist_ok=True)\n    # py_vncorenlp changes cwd; all paths are absolute and cwd is restored.\n    import os\n    previous_cwd = Path.cwd()\n    try:\n        segmenter = py_vncorenlp.VnCoreNLP(max_heap_size=\'-Xmx1g\', annotators=[\'wseg\'], save_dir=str(model_dir))\n    finally:\n        os.chdir(previous_cwd)\n    preserve_java_unicode(segmenter)\n    result = {}\n    for split, frame in frames.items():\n        frame = frame.copy()\n        frame[\'text_normalized\'] = frame.text_raw.map(normalize_text)\n        segmented = []\n        for row in tqdm(frame.itertuples(), total=len(frame), desc=f\'Segment {split}\'):\n            try:\n                text = \' \'.join(segmenter.word_segment(row.text_normalized)).strip()\n                if not text:\n                    raise ValueError(\'Empty segmenter output\')\n                if supplementary_sequence(row.text_normalized) != supplementary_sequence(text):\n                    raise ValueError(\'Supplementary Unicode changed during segmentation\')\n            except Exception as exc:\n                raise RuntimeError(f\'Segmentation failed for {row.sample_id}\') from exc\n            segmented.append(text)\n        frame[\'text_segmented\'] = segmented\n        frame[\'preprocess_version\'] = PREPROCESS_VERSION\n        frame.to_parquet(target / f\'{split}.parquet\', index=False)\n        result[split] = frame\n    dump_json(metadata, signature)\n    dump_json(root / \'data/processed/latest.json\', {\'path\': str(target), \'fingerprint\': signature_hash})\n    sample = pd.concat([result[\'train\'].query(\'label == @label\').sample(30, random_state=42) for label in range(3)])\n    sample.to_csv(root / \'reports/preprocessing_train_examples.csv\', index=False)\n    return result\n', 'vihsd/eda.py': '"""EDA artifacts with counts, train-only inspection and an HTML review report."""\nfrom __future__ import annotations\nimport html\nimport re\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.metrics import accuracy_score, f1_score\nfrom .data import LABELS, dump_json, normalize_text\n\nPALETTE = [\'#3979a8\', \'#d79c34\', \'#bd5066\']\n\n\ndef features(frames):\n    result = []\n    for split, frame in frames.items():\n        f = frame[[\'sample_id\', \'split\', \'label\']].copy()\n        text = frame.text_raw.fillna(\'\').astype(str)\n        f[\'characters\'] = text.str.len()\n        f[\'whitespace_units\'] = text.str.split().str.len()\n        f[\'has_emoji\'] = text.str.contains(r\'[\\U0001F000-\\U0001FAFF\\u2600-\\u27BF]\', regex=True)\n        f[\'has_url\'] = text.str.contains(r\'https?://|www\\.\', case=False, regex=True)\n        f[\'has_mention_marker\'] = text.str.contains(r\'@\\w+|<person\\s*name>\', case=False, regex=True)\n        f[\'has_hashtag\'] = text.str.contains(r\'#\\w+\', regex=True)\n        f[\'has_digit\'] = text.str.contains(r\'\\d\', regex=True)\n        f[\'has_elongation\'] = text.map(lambda t: bool(re.search(r\'([^\\W\\d_])\\1{2,}\', t, re.IGNORECASE)))\n        f[\'has_uppercase\'] = text.map(lambda t: any(c.isupper() for c in t))\n        f[\'empty_after_normalization\'] = text.map(lambda t: not normalize_text(t))\n        result.append(f)\n    return pd.concat(result, ignore_index=True)\n\n\ndef create_eda(frames, output, audit_summary, token_lengths=None, token_frames=None):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    sns.set_theme(style=\'whitegrid\', context=\'notebook\', font=\'DejaVu Sans\')\n    plt.rcParams.update({\'figure.dpi\': 120, \'savefig.dpi\': 160, \'axes.titleweight\': \'bold\'})\n    f = features(frames)\n    f.to_csv(output / \'sample_features.csv\', index=False)\n    counts = pd.crosstab(f.split, f.label).reindex(index=[\'train\', \'validation\', \'test\'], columns=[0, 1, 2], fill_value=0)\n    counts.columns = LABELS\n    counts.to_csv(output / \'label_counts.csv\')\n    proportions = counts.div(counts.sum(axis=1), axis=0)\n    proportions.to_csv(output / \'label_proportions.csv\')\n    quantiles = f.groupby([\'split\', \'label\'])[[\'characters\', \'whitespace_units\']].quantile([0, .25, .5, .75, .9, .95, .99, 1])\n    quantiles.to_csv(output / \'length_quantiles.csv\')\n    ugc_columns = [c for c in f if c.startswith(\'has_\')]\n    ugc = f.groupby(\'split\')[ugc_columns].mean().reindex([\'train\', \'validation\', \'test\'])\n    ugc.to_csv(output / \'ugc_rates.csv\')\n    figures = []\n\n    def save(name, fig):\n        fig.savefig(output / name, bbox_inches=\'tight\')\n        plt.close(fig)\n        figures.append(name)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    counts.plot.bar(ax=axes[0], color=PALETTE, rot=0)\n    proportions.plot.bar(ax=axes[1], stacked=True, color=PALETTE, rot=0)\n    axes[0].set(title=\'Số mẫu theo nhãn và split\', ylabel=\'Số mẫu\', xlabel=\'\')\n    axes[1].set(title=\'Tỷ lệ nhãn theo split\', ylabel=\'Tỷ lệ\', xlabel=\'\', ylim=(0, 1))\n    axes[1].legend(loc=\'lower right\')\n    save(\'01_labels.png\', fig)\n\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))\n    for split, color in zip([\'train\', \'validation\', \'test\'], PALETTE):\n        subset = f[f.split == split]\n        sns.histplot(data=subset, x=\'characters\', bins=np.linspace(0, 1000, 51), stat=\'probability\',\n                     element=\'step\', fill=False, label=split, color=color, ax=axes[0])\n        sns.ecdfplot(data=subset, x=\'whitespace_units\', label=split, color=color, ax=axes[1])\n    axes[0].set(title=\'Độ dài ký tự (khung nhìn 0–1.000)\', xlabel=\'Số ký tự\', ylabel=\'Tỷ lệ mẫu\')\n    axes[0].legend()\n    axes[1].set(title=\'Phân phối tích lũy độ dài\', xlabel=\'Đơn vị cách nhau bằng khoảng trắng\', ylabel=\'Tỷ lệ tích lũy\', xlim=(0, 200))\n    axes[1].legend()\n    save(\'02_lengths.png\', fig)\n\n    fig, ax = plt.subplots(figsize=(9, 4.8))\n    sns.boxplot(data=f[f.split == \'train\'], x=\'label\', y=\'whitespace_units\', hue=\'label\',\n                palette=PALETTE, showfliers=False, legend=False, ax=ax)\n    ax.set(xticks=[0, 1, 2], xticklabels=LABELS, xlabel=\'Nhãn\', ylabel=\'Đơn vị cách nhau bằng khoảng trắng\',\n           title=\'Độ dài theo nhãn — train (ẩn outlier để dễ đọc)\')\n    save(\'03_length_by_class.png\', fig)\n\n    fig, ax = plt.subplots(figsize=(11, 4.5))\n    sns.heatmap(ugc * 100, annot=True, fmt=\'.1f\', cmap=\'Blues\', ax=ax, cbar_kws={\'label\': \'% mẫu\'})\n    ax.set(title=\'Tín hiệu bề mặt UGC — tỷ lệ mẫu (%)\', xlabel=\'\', ylabel=\'\')\n    ax.set_xticklabels([\'Emoji\', \'URL\', \'Mention marker\', \'Hashtag\', \'Chữ số\', \'Lặp ký tự\', \'Chữ hoa\'], rotation=20, ha=\'right\')\n    save(\'04_ugc.png\', fig)\n\n    majority = int(frames[\'train\'].label.value_counts().idxmax())\n    baseline = {}\n    for split, frame in frames.items():\n        predicted = np.full(len(frame), majority)\n        baseline[split] = {\'majority_label\': majority, \'accuracy\': accuracy_score(frame.label, predicted),\n                           \'macro_f1\': f1_score(frame.label, predicted, labels=[0, 1, 2], average=\'macro\', zero_division=0)}\n    dump_json(output / \'majority_baseline.json\', baseline)\n    examples = pd.concat([frames[\'train\'].query(\'label == @label\').sample(30, random_state=42) for label in [0, 1, 2]])\n    examples.to_csv(output / \'train_review_sample.csv\', index=False)\n    if token_lengths is not None:\n        token_df, token_summary = token_diagnostics(token_frames if token_frames is not None else frames, token_lengths, output)\n        fig, ax = plt.subplots(figsize=(10, 4.8))\n        for split, color in zip([\'train\', \'validation\', \'test\'], PALETTE):\n            sns.ecdfplot(data=token_df[token_df.split == split], x=\'bpe_tokens\', label=split, color=color, ax=ax)\n        for threshold in [100, 128, 256]:\n            ax.axvline(threshold, color=\'#8b9299\', linestyle=\'--\', linewidth=1)\n        ax.set(title=\'Độ dài BPE trước truncation (gồm special tokens)\', xlabel=\'Số token\', ylabel=\'Tỷ lệ tích lũy\', xlim=(0, 400))\n        ax.legend()\n        save(\'05_bpe_lengths.png\', fig)\n    else:\n        token_summary = None\n    raw_dup = audit_summary[\'duplicates\'][\'normalized\']\n    empty_n = int(f.empty_after_normalization.sum())\n    train_clean_rate = proportions.loc[\'train\', \'CLEAN\'] * 100\n    text = f\'\'\'# EDA ViHSD — baseline PhoBERT phục vụ robustness\n\n## Dữ liệu và phạm vi\n\nNguồn: archive trong GitHub chính thức `sonlam1102/vihsd`, phiên bản/checksum trong manifest.\nGiữ nguyên split chính thức. Test gốc nghĩa là chưa thêm nhiễu nhân tạo; vẫn có lỗi gõ và UGC tự nhiên.\nCác thống kê test chỉ mô tả dữ liệu, không dùng để lựa chọn siêu tham số hoặc sửa preprocessing.\n\n## Phát hiện chính\n\n- Tổng **{len(f):,}** mẫu gốc; train **{len(frames[\'train\']):,}**, validation **{len(frames[\'validation\']):,}**, test **{len(frames[\'test\']):,}**.\n- CLEAN chiếm **{train_clean_rate:.2f}%** train. Accuracy cao có thể do dự đoán lớp đa số; chọn **Macro-F1** làm chỉ số chính.\n- Baseline luôn đoán CLEAN trên validation: accuracy **{baseline[\'validation\'][\'accuracy\']:.4f}**, Macro-F1 **{baseline[\'validation\'][\'macro_f1\']:.4f}**.\n- Có **{empty_n}** dòng rỗng sau chuẩn hóa kỹ thuật. Hai dòng rỗng thuộc train được ghi vào exclusion ledger; raw và split đánh giá giữ nguyên.\n- Sau NFC/khoảng trắng: **{raw_dup[\'cross_split_groups\']:,} nhóm trùng giữa các split**, **{raw_dup[\'conflicting_label_groups\']:,} nhóm cùng văn bản khác nhãn** trên toàn bộ dữ liệu. Đây là giới hạn quan trọng của đánh giá split chính thức; không tự deduplicate.\n- Emoji/chữ hoa/ký tự lặp là tín hiệu bề mặt, không phải nhãn độc hại. Giữ nguyên để baseline phản ánh UGC.\n\n## Quyết định pipeline\n\n1. Giữ nguyên train/validation/test chính thức; chỉ loại hai mẫu train không có nội dung theo ledger có ID cụ thể.\n2. Unicode NFC và khoảng trắng; giữ dấu, case, emoji, URL, hashtag, teencode và ký tự kéo dài.\n3. Truyền UTF-8 qua cầu nối Java để bảo toàn emoji/non-BMP, rồi RDRSegmenter và PhoBERT BPE; không xóa stopword, không sửa chính tả bằng từ điển.\n4. `max_length=256` gồm special tokens. Thống kê truncation dưới đây dùng để ghi nhận mất thông tin; cấu hình được định trước và không điều chỉnh theo test.\n5. Cross-entropy thường, không resampling/class weight ở lần chạy đầu. Báo cáo precision/recall/F1 theo từng lớp.\n\n## Cách đọc số liệu\n\nĐộ dài theo khoảng trắng là số đơn vị phân cách, không phải số từ tiếng Việt sau segmentation.\nMention marker chỉ nhận `@...` hoặc placeholder, không suy ra được toàn bộ tên người được mention.\nEmoji và lặp ký tự được nhận dạng heuristic nên không phải annotation chuẩn.\nKhông suy luận quan hệ nhân quả giữa segmentation và sai nhãn từ EDA.\n\'\'\'\n    if token_summary:\n        text += \'\\n## Tokenization thực tế\\n\\n\'\n        for split, stats in token_summary.items():\n            text += f"- **{split}**: median {stats[\'p50\']:.0f}, p95 {stats[\'p95\']:.0f}, p99 {stats[\'p99\']:.0f} token; cắt ở 256: {stats[\'truncated_at_256_rate\']*100:.2f}%; `<unk>` trên token nội dung: {stats[\'unk_token_rate\']*100:.4f}%.\\n"\n        text += \'\\nKhông đồng nhất `<unk>` với phân mảnh: hai chỉ số được đo riêng trong bảng tokenization.\\n\'\n    for name in figures:\n        text += f\'\\n![{name}]({name})\\n\'\n    (output / \'EDA_REPORT.md\').write_text(text, encoding=\'utf-8\')\n    html_parts = ["<!doctype html><meta charset=\'utf-8\'><title>EDA ViHSD</title><style>body{font:16px system-ui;max-width:1100px;margin:40px auto;color:#203040;line-height:1.6}img{width:100%;margin:20px 0}pre{white-space:pre-wrap}table{border-collapse:collapse}td,th{padding:8px;border:1px solid #ddd}</style>",\n                  \'<h1>EDA ViHSD — baseline PhoBERT</h1>\', \'<pre>\' + html.escape(text.split(\'![\', 1)[0]) + \'</pre>\', counts.to_html()]\n    html_parts += [f\'<img src="{name}" alt="{name}">\' for name in figures]\n    (output / \'index.html\').write_text(\'\\n\'.join(html_parts), encoding=\'utf-8\')\n    return {\'n\': len(f), \'majority_baseline\': baseline, \'tokenization\': token_summary, \'figures\': figures}\n\n\ndef token_diagnostics(frames, token_data, output):\n    """token_data: split -> lengths, unk_counts, content_counts (all before truncation)."""\n    rows = []\n    summary = {}\n    for split, frame in frames.items():\n        details = token_data[split]\n        lengths = np.asarray(details[\'lengths\'])\n        assert len(lengths) == len(frame)\n        units = frame.text_segmented.str.split().str.len().clip(lower=1).to_numpy()\n        df = frame[[\'sample_id\', \'split\', \'label\']].copy()\n        df[\'bpe_tokens\'] = lengths\n        df[\'bpe_per_segmented_unit\'] = np.asarray(details[\'content_counts\']) / units\n        df[\'unk_tokens\'] = details[\'unk_counts\']\n        rows.append(df)\n        stats = {\'p50\': float(np.quantile(lengths, .5)), \'p95\': float(np.quantile(lengths, .95)),\n                 \'p99\': float(np.quantile(lengths, .99)), \'max\': int(lengths.max()),\n                 \'mean_bpe_per_segmented_unit\': float(df.bpe_per_segmented_unit.mean()),\n                 \'unk_token_rate\': float(sum(details[\'unk_counts\']) / max(1, sum(details[\'content_counts\'])))}\n        for threshold in [100, 128, 256]:\n            stats[f\'truncated_at_{threshold}_rate\'] = float((lengths > threshold).mean())\n        summary[split] = stats\n    data = pd.concat(rows, ignore_index=True)\n    data.to_csv(Path(output) / \'token_lengths.csv\', index=False)\n    dump_json(Path(output) / \'tokenization_summary.json\', summary)\n    return data, summary\n', 'vihsd/pipeline.py': '"""Shared stages used by CLI and embedded Kaggle notebook."""\nfrom __future__ import annotations\nimport json\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\nfrom .data import audit, download, dump_json, load_raw, prepare, sha256_bytes\nfrom .eda import create_eda\n\n\ndef ensure_java():\n    """Use the installed JVM; pyjnius binary wheels don\'t need javac at runtime."""\n    if not shutil.which(\'java\'):\n        raise RuntimeError(\'Java runtime is required for RDRSegmenter\')\n    home = Path(shutil.which(\'java\')).resolve().parent.parent\n    os.environ.setdefault(\'JAVA_HOME\', str(home))\n    os.environ.setdefault(\'JDK_HOME\', str(home))\n    return subprocess.check_output([\'java\', \'-version\'], stderr=subprocess.STDOUT, text=True)\n\n\ndef prepare_all(root, config):\n    from transformers import AutoTokenizer\n    from .training import resolve_config, tokenize_frames\n    root = Path(root).resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    print(\'STAGE: DATA_AUDIT\', flush=True)\n    raw = load_raw(root)\n    audit_summary = audit(raw, root / \'reports/audit\')\n    # Write raw EDA before potentially expensive Java/model downloads.\n    create_eda(raw, root / \'reports/eda\', audit_summary)\n    print(\'STAGE: SEGMENTATION\', flush=True)\n    java_version = ensure_java()\n    processed = prepare(root)\n    config = resolve_config(config)\n    print(\'MODEL_REVISION:\', config[\'model_revision\'], flush=True)\n    tokenizer = AutoTokenizer.from_pretrained(config[\'model_name\'], revision=config[\'model_revision\'], use_fast=False)\n    assert config[\'max_length\'] <= tokenizer.model_max_length\n    print(\'STAGE: TOKENIZATION_AND_EDA\', flush=True)\n    encoded, diagnostics = tokenize_frames(processed, tokenizer, config[\'max_length\'], root / \'data/tokenized\')\n    create_eda(raw, root / \'reports/eda\', audit_summary, token_lengths=diagnostics, token_frames=processed)\n    dump_json(root / \'reports/java.json\', {\'java_version\': java_version})\n    dump_json(root / \'reports/resolved_config.json\', config)\n    source_dir = Path(__file__).parent\n    source_hashes = {p.name: sha256_bytes(p.read_bytes()) for p in source_dir.glob(\'*.py\')}\n    dump_json(root / \'reports/provenance.json\', {\'source_sha256\': source_hashes,\n              \'data\': json.loads((root / \'data/raw/manifest.json\').read_text()),\n              \'processed\': json.loads((root / \'data/processed/latest.json\').read_text()),\n              \'config\': config})\n    print(\'STAGE: DATA_READY\', {s: len(f) for s, f in processed.items()}, flush=True)\n    return processed, encoded, tokenizer, config\n\n\ndef run_training(root, config, stages=None):\n    from .training import smoke_and_overfit, train, evaluate_checkpoint\n    root = Path(root).resolve()\n    frames, encoded, tokenizer, config = stages or prepare_all(root, config)\n    print(\'STAGE: SMOKE_AND_OVERFIT\', flush=True)\n    checks = smoke_and_overfit(frames, encoded, tokenizer, config, root / \'reports/checks\')\n    run_dir = root / \'runs\' / config[\'run_name\']\n    run_dir.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(root / \'reports/provenance.json\', run_dir / \'provenance.json\')\n    print(\'STAGE: FULL_TRAINING\', flush=True)\n    best = train(frames, encoded, tokenizer, config, run_dir)\n    print(\'STAGE: FINAL_TEST\', flush=True)\n    result = evaluate_checkpoint(best, frames, encoded, tokenizer, run_dir / \'evaluation\', config[\'eval_batch_size\'])\n    write_baseline_report(root, run_dir, result)\n    dump_json(root / \'RUN_COMPLETE.json\', {\'run_dir\': str(run_dir), \'test_metrics\': result, \'checks\': checks})\n    print(\'RUN_COMPLETE\', flush=True)\n    return result\n\n\ndef write_baseline_report(root, run_dir, result):\n    history = json.loads((run_dir / \'history.json\').read_text())\n    best = json.loads((run_dir / \'best/selection.json\').read_text())\n    majority = json.loads((root / \'reports/eda/majority_baseline.json\').read_text())[\'test\']\n    text = f\'\'\'# Baseline PhoBERT — lần training đầu tiên\n\nĐây là baseline mới phục vụ nghiên cứu robustness, không phải tái lập điểm PhoBERT trong bảng kết quả bài ViHSD 2021.\n\n- Checkpoint chọn bằng **validation Macro-F1**, epoch **{best[\'epoch\']}**, score **{best[\'score\']:.4f}**.\n- Test gốc: **Macro-F1 {result[\'macro_f1\']:.4f}**, **accuracy {result[\'accuracy\']:.4f}**.\n- Majority baseline test: Macro-F1 {majority[\'macro_f1\']:.4f}, accuracy {majority[\'accuracy\']:.4f}.\n- Số epoch hoàn thành: {len(history)}. Một seed 42, chưa đánh giá độ biến thiên qua nhiều seed.\n- Train gốc 24.048 mẫu; loại 2 mẫu rỗng có ID được ghi trong audit, học trên 24.046 mẫu. Validation/test giữ nguyên.\n- Giữ nguyên duplicate trong split chính thức. Trùng giữa các split và xung đột nhãn là giới hạn của kết quả này.\n- Không sửa teencode, bỏ dấu, xóa emoji hay stopword. Chưa thêm nhiễu nhân tạo và chưa áp dụng FGM.\n\n## Chỉ số từng lớp\n\n| Nhãn | Precision | Recall | F1 | Support |\n|---|---:|---:|---:|---:|\n\'\'\'\n    for label in [\'CLEAN\', \'OFFENSIVE\', \'HATE\']:\n        c = result[\'per_class\'][label]\n        text += f"| {label} | {c[\'precision\']:.4f} | {c[\'recall\']:.4f} | {c[\'f1-score\']:.4f} | {c[\'support\']:.0f} |\\n"\n    text += \'\\n## Hướng đọc tiếp\\n\\nXem confusion matrix và error_review.csv để mô tả lỗi. Không suy luận segmentation gây lỗi nếu chưa làm ablation. Đợt tiếp theo: cố định checkpoint và xây bộ nhiễu có kiểm tra bảo toàn nhãn.\\n\'\n    (run_dir / \'BASELINE_REPORT.md\').write_text(text, encoding=\'utf-8\')\n', 'vihsd/training.py': '"""Single-GPU PyTorch baseline; preprocessing and test never enter model selection."""\nfrom __future__ import annotations\nimport copy\nimport gc\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\nfrom .data import LABELS, dump_json, sha256_bytes\n\n\ndef metrics(y_true, y_pred):\n    return {\'accuracy\': float(accuracy_score(y_true, y_pred)),\n            \'macro_f1\': float(f1_score(y_true, y_pred, labels=[0, 1, 2], average=\'macro\', zero_division=0)),\n            \'per_class\': classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=LABELS, zero_division=0, output_dict=True),\n            \'confusion_matrix\': confusion_matrix(y_true, y_pred, labels=[0, 1, 2]).tolist()}\n\n\ndef seed_all(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\n\ndef resolve_config(config):\n    from huggingface_hub import model_info\n    resolved = copy.deepcopy(config)\n    resolved[\'model_revision\'] = model_info(config[\'model_name\'], revision=config.get(\'model_revision\', \'main\')).sha\n    return resolved\n\n\ndef tokenize_frames(frames, tokenizer, max_length, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    encoded, diagnostics = {}, {}\n    for split, frame in frames.items():\n        ids, masks, lengths, unks, contents = [], [], [], [], []\n        texts = frame.text_segmented.tolist()\n        for start in range(0, len(texts), 256):\n            batch = tokenizer(texts[start:start+256], add_special_tokens=True, truncation=False, padding=False,\n                              return_attention_mask=False, verbose=False)[\'input_ids\']\n            for tokens in batch:\n                lengths.append(len(tokens))\n                content = tokens[1:-1]\n                contents.append(len(content))\n                unks.append(content.count(tokenizer.unk_token_id))\n                # Use the tokenizer\'s own truncation path to preserve EOS.\n            truncated = tokenizer(texts[start:start+256], truncation=True, padding=False, max_length=max_length)\n            ids.extend(truncated[\'input_ids\'])\n            masks.extend(truncated[\'attention_mask\'])\n        encoded[split] = [{\'input_ids\': i, \'attention_mask\': m, \'labels\': int(y)}\n                          for i, m, y in zip(ids, masks, frame.label)]\n        diagnostics[split] = {\'lengths\': lengths, \'unk_counts\': unks, \'content_counts\': contents}\n        assert len(encoded[split]) == len(frame)\n        assert all(2 <= len(x[\'input_ids\']) <= max_length for x in encoded[split])\n        # Compact non-executable cache for replay and inspection.\n        dump_json(output / f\'{split}.json\', encoded[split])\n    dump_json(output / \'token_diagnostics.json\', diagnostics)\n    return encoded, diagnostics\n\n\ndef make_model(config):\n    from transformers import AutoModelForSequenceClassification\n    return AutoModelForSequenceClassification.from_pretrained(\n        config[\'model_name\'], revision=config[\'model_revision\'], num_labels=3,\n        id2label=dict(enumerate(LABELS)), label2id={v: k for k, v in enumerate(LABELS)})\n\n\ndef make_loader(records, tokenizer, batch_size, shuffle=False, seed=42):\n    import torch\n    from transformers import DataCollatorWithPadding\n    return torch.utils.data.DataLoader(records, batch_size=batch_size, shuffle=shuffle,\n        generator=torch.Generator().manual_seed(seed), num_workers=0,\n        collate_fn=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8, return_tensors=\'pt\'),\n        pin_memory=torch.cuda.is_available())\n\n\ndef predict(model, records, tokenizer, batch_size, device):\n    import torch\n    was_training = model.training\n    model.eval()\n    logits, labels, losses = [], [], []\n    with torch.inference_mode():\n        for batch in make_loader(records, tokenizer, batch_size):\n            batch = {k: v.to(device) for k, v in batch.items()}\n            result = model(**batch)\n            losses.append((float(result.loss), len(batch[\'labels\'])))\n            logits.append(result.logits.float().cpu())\n            labels.extend(batch[\'labels\'].cpu().tolist())\n    all_logits = torch.cat(logits)\n    probabilities = all_logits.softmax(dim=-1).numpy()\n    result = metrics(labels, probabilities.argmax(axis=1))\n    result[\'loss\'] = sum(x*n for x, n in losses) / sum(n for _, n in losses)\n    if was_training:\n        model.train()\n    return result, probabilities\n\n\ndef balanced_indices(frame, per_class, seed=42):\n    return np.concatenate([frame.index[frame.label == label].to_series().sample(min(per_class, int((frame.label == label).sum())), random_state=seed).to_numpy() for label in range(3)]).astype(int)\n\n\ndef smoke_and_overfit(frames, encoded, tokenizer, config, output):\n    """Real pretrained weights: finite backward, roundtrip, then learn 48 train examples."""\n    import torch\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    seed_all(config[\'seed\'])\n    idx = balanced_indices(frames[\'train\'], 16)\n    records = [encoded[\'train\'][i] for i in idx]\n    model = make_model(config).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)\n    loader = make_loader(records, tokenizer, 8, shuffle=True)\n    model.train()\n    losses = []\n    for step, batch in enumerate(loader):\n        batch = {k: v.to(device) for k, v in batch.items()}\n        loss = model(**batch).loss\n        assert torch.isfinite(loss), \'Nonfinite smoke loss\'\n        loss.backward()\n        assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n        optimizer.step()\n        optimizer.zero_grad(set_to_none=True)\n        losses.append(float(loss.detach()))\n        if step == 2:\n            break\n    model.eval()\n    reference = next(iter(make_loader(records[:4], tokenizer, 4)))\n    reference = {k: v.to(device) for k, v in reference.items()}\n    with torch.no_grad():\n        expected = model(**reference).logits.cpu()\n    model.save_pretrained(output / \'roundtrip\')\n    # Optimizer state and RNG roundtrip: trusted checkpoint generated by this run.\n    torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'torch_rng\': torch.get_rng_state(),\n                \'numpy_rng\': np.random.get_state(), \'python_rng\': random.getstate()}, output / \'smoke_resume.pt\')\n    saved = torch.load(output / \'smoke_resume.pt\', map_location=device, weights_only=False)\n    model.load_state_dict(saved[\'model\'])\n    optimizer.load_state_dict(saved[\'optimizer\'])\n    torch.set_rng_state(saved[\'torch_rng\'].cpu())\n    np.random.set_state(saved[\'numpy_rng\'])\n    random.setstate(saved[\'python_rng\'])\n    del saved\n    reloaded = type(model).from_pretrained(output / \'roundtrip\').to(device).eval()\n    with torch.no_grad():\n        actual = reloaded(**reference).logits.cpu()\n    max_diff = float((expected - actual).abs().max())\n    assert max_diff < 1e-5, f\'Roundtrip mismatch: {max_diff}\'\n    del reloaded\n    gc.collect()\n    # Start the overfit check with a fresh optimizer, in FP32; this model is discarded.\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)\n    steps = 0\n    result = None\n    while steps < 300:\n        model.train()\n        for batch in make_loader(records, tokenizer, 8, shuffle=True, seed=config[\'seed\'] + steps):\n            batch = {k: v.to(device) for k, v in batch.items()}\n            loss = model(**batch).loss\n            if not torch.isfinite(loss):\n                raise RuntimeError(\'Nonfinite overfit loss\')\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)\n            optimizer.step()\n            optimizer.zero_grad(set_to_none=True)\n            steps += 1\n        if steps % 30 == 0:\n            result, _ = predict(model, records, tokenizer, 8, device)\n            print(f\'OVERFIT step={steps} accuracy={result["accuracy"]:.4f}\', flush=True)\n            if result[\'accuracy\'] >= .95:\n                break\n    if result is None:\n        result, _ = predict(model, records, tokenizer, 8, device)\n    summary = {\'smoke_losses\': losses, \'roundtrip_max_logit_difference\': max_diff,\n               \'optimizer_state_roundtrip\': True, \'overfit_steps\': steps, \'overfit_metrics\': result,\n               \'passed\': result[\'accuracy\'] >= .95}\n    dump_json(output / \'checks.json\', summary)\n    del model, optimizer\n    gc.collect()\n    if device.type == \'cuda\':\n        torch.cuda.empty_cache()\n    # Remove diagnostic model files; only the final baseline needs full weights.\n    import shutil\n    shutil.rmtree(output / \'roundtrip\')\n    (output / \'smoke_resume.pt\').unlink()\n    if not summary[\'passed\']:\n        raise RuntimeError(\'Overfit check failed; full training blocked. See checks.json\')\n    return summary\n\n\ndef environment():\n    import importlib.metadata\n    import torch\n    versions = {}\n    for package in [\'torch\', \'transformers\', \'accelerate\', \'pandas\', \'numpy\', \'py-vncorenlp\', \'tokenizers\', \'scikit-learn\']:\n        versions[package] = importlib.metadata.version(package)\n    return {\'python\': platform.python_version(), \'packages\': versions,\n            \'cuda\': torch.version.cuda, \'gpu\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            \'visible_gpu_count\': torch.cuda.device_count()}\n\n\ndef train(frames, encoded, tokenizer, config, run_dir, resume=None):\n    import torch\n    from transformers import get_linear_schedule_with_warmup\n    run_dir = Path(run_dir)\n    run_dir.mkdir(parents=True, exist_ok=True)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Full training requires CUDA; use smoke tests for local CPU checks\')\n    device = torch.device(\'cuda:0\')\n    seed_all(config[\'seed\'])\n    model = make_model(config).to(device)\n    batch_size = config[\'train_batch_size\']\n    accum = config[\'gradient_accumulation_steps\']\n    # Probe longest training records, preserve the intended effective batch if OOM.\n    longest = sorted(encoded[\'train\'], key=lambda r: len(r[\'input_ids\']), reverse=True)\n    while True:\n        try:\n            batch = next(iter(make_loader(longest[:batch_size], tokenizer, batch_size)))\n            loss = model(**{k: v.to(device) for k, v in batch.items()}).loss\n            loss.backward()\n            model.zero_grad(set_to_none=True)\n            del loss, batch\n            break\n        except torch.cuda.OutOfMemoryError:\n            model.zero_grad(set_to_none=True)\n            gc.collect()\n            torch.cuda.empty_cache()\n            if batch_size <= 2:\n                if model.is_gradient_checkpointing:\n                    raise\n                model.gradient_checkpointing_enable()\n            else:\n                batch_size //= 2\n                accum *= 2\n    config = {**config, \'actual_train_batch_size\': batch_size, \'actual_gradient_accumulation_steps\': accum,\n              \'effective_batch_size\': batch_size * accum, \'gradient_checkpointing\': model.is_gradient_checkpointing}\n    config[\'precision\'] = \'bf16\' if torch.cuda.is_bf16_supported(including_emulation=False) else \'fp16\'\n    dump_json(run_dir / \'config.json\', config)\n    dump_json(run_dir / \'environment.json\', environment())\n    n_batches = math.ceil(len(encoded[\'train\']) / batch_size)\n    steps_per_epoch = math.ceil(n_batches / accum)\n    total_steps = steps_per_epoch * config[\'epochs\']\n    no_decay = (\'bias\', \'LayerNorm.weight\', \'layer_norm.weight\')\n    parameters = [\n        {\'params\': [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], \'weight_decay\': config[\'weight_decay\']},\n        {\'params\': [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], \'weight_decay\': 0.0}]\n    optimizer = torch.optim.AdamW(parameters, lr=config[\'learning_rate\'], betas=(.9, .999), eps=1e-8)\n    scheduler = get_linear_schedule_with_warmup(optimizer, math.ceil(total_steps * config[\'warmup_ratio\']), total_steps)\n    use_bf16 = config[\'precision\'] == \'bf16\'\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=not use_bf16)\n    start_epoch, skip_batches, global_step, best_score = 0, 0, 0, -1.0\n    best_epoch = None\n    history = []\n    if resume:\n        saved = torch.load(resume, map_location=\'cpu\', weights_only=False)\n        if saved[\'config\'] != config:\n            raise ValueError(\'Resume config differs from checkpoint\')\n        model.load_state_dict(saved[\'model\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        start_epoch, skip_batches = saved[\'next_epoch\'], saved[\'next_batch\']\n        global_step, best_score, best_epoch = saved[\'global_step\'], saved[\'best_score\'], saved[\'best_epoch\']\n        history = saved[\'history\']\n        random.setstate(saved[\'python_rng\']); np.random.set_state(saved[\'numpy_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\']); torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        del saved\n    started = time.time()\n    torch.cuda.reset_peak_memory_stats()\n\n    def checkpoint(next_epoch, next_batch):\n        state = {\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                 \'scaler\': scaler.state_dict(), \'config\': config, \'next_epoch\': next_epoch, \'next_batch\': next_batch,\n                 \'global_step\': global_step, \'best_score\': best_score, \'best_epoch\': best_epoch, \'history\': history,\n                 \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(), \'torch_rng\': torch.get_rng_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all()}\n        tmp = run_dir / \'last.tmp\'\n        torch.save(state, tmp)\n        tmp.replace(run_dir / \'last.pt\')\n\n    print(f\'TRAINING_STARTED examples={len(encoded["train"])} total_steps={total_steps} gpu={torch.cuda.get_device_name(0)} precision={config["precision"]}\', flush=True)\n    for epoch in range(start_epoch, config[\'epochs\']):\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        epoch_loss, epoch_n = 0., 0\n        loader = make_loader(encoded[\'train\'], tokenizer, batch_size, shuffle=True, seed=config[\'seed\'] + epoch)\n        for batch_idx, batch in enumerate(loader):\n            if epoch == start_epoch and batch_idx < skip_batches:\n                continue\n            batch = {k: v.to(device) for k, v in batch.items()}\n            group_start = (batch_idx // accum) * accum\n            group_size = min(accum, n_batches - group_start)\n            with torch.autocast(\'cuda\', dtype=torch.bfloat16 if use_bf16 else torch.float16):\n                loss = model(**batch).loss\n            if not torch.isfinite(loss):\n                checkpoint(epoch, group_start)\n                raise RuntimeError(f\'Nonfinite loss at epoch {epoch}, batch {batch_idx}\')\n            scaler.scale(loss / group_size).backward()\n            n = len(batch[\'labels\'])\n            epoch_loss += float(loss.detach()) * n\n            epoch_n += n\n            if (batch_idx + 1) % accum == 0 or batch_idx + 1 == n_batches:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), config[\'max_grad_norm\'])\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() >= old_scale:\n                    scheduler.step()\n                optimizer.zero_grad(set_to_none=True)\n                global_step += 1\n                if global_step == 1 or global_step % 50 == 0:\n                    record = {\'epoch\': epoch + 1, \'step\': global_step, \'train_loss\': epoch_loss / max(epoch_n, 1),\n                              \'learning_rate\': scheduler.get_last_lr()[0], \'elapsed_seconds\': time.time() - started}\n                    with (run_dir / \'steps.jsonl\').open(\'a\') as f:\n                        f.write(json.dumps(record) + \'\\n\')\n                    print(\'TRAIN_STEP \' + json.dumps(record), flush=True)\n                if global_step % config[\'save_steps\'] == 0:\n                    checkpoint(epoch, batch_idx + 1)\n        validation, _ = predict(model, encoded[\'validation\'], tokenizer, config[\'eval_batch_size\'], device)\n        record = {\'epoch\': epoch + 1, \'step\': global_step, \'train_loss\': epoch_loss / max(epoch_n, 1),\n                  \'validation\': validation, \'elapsed_seconds\': time.time() - started,\n                  \'peak_gpu_memory_gb\': torch.cuda.max_memory_allocated() / 1024**3}\n        history.append(record)\n        dump_json(run_dir / \'history.json\', history)\n        print(\'VALIDATION \' + json.dumps(record), flush=True)\n        if validation[\'macro_f1\'] > best_score:\n            best_score, best_epoch = validation[\'macro_f1\'], epoch + 1\n            model.save_pretrained(run_dir / \'best\')\n            tokenizer.save_pretrained(run_dir / \'best\')\n            dump_json(run_dir / \'best/selection.json\', {\'split\': \'validation\', \'metric\': \'macro_f1\', \'score\': best_score, \'epoch\': best_epoch})\n        checkpoint(epoch + 1, 0)\n    dump_json(run_dir / \'training_summary.json\', {\'best_validation_macro_f1\': best_score, \'best_epoch\': best_epoch,\n                                               \'global_step\': global_step, \'elapsed_seconds\': time.time() - started})\n    del model, optimizer, scheduler, scaler\n    gc.collect()\n    torch.cuda.empty_cache()\n    return run_dir / \'best\'\n\n\ndef evaluate_checkpoint(checkpoint, frames, encoded, tokenizer, output, batch_size=16):\n    import torch\n    import matplotlib.pyplot as plt\n    import seaborn as sns\n    from transformers import AutoModelForSequenceClassification\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device).eval()\n    result, probabilities = predict(model, encoded[\'test\'], tokenizer, batch_size, device)\n    predictions = frames[\'test\'][[\'sample_id\', \'label\']].copy().reset_index(drop=True)\n    predictions[\'prediction\'] = probabilities.argmax(axis=1)\n    for i, label in enumerate(LABELS):\n        predictions[f\'p_{label.lower()}\'] = probabilities[:, i]\n    assert len(predictions) == len(frames[\'test\']) and predictions.sample_id.is_unique\n    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(1), 1, atol=1e-5)\n    predictions.to_parquet(output / \'predictions.parquet\', index=False)\n    dump_json(output / \'metrics.json\', result)\n    cm = np.asarray(result[\'confusion_matrix\'])\n    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(output / \'confusion_matrix.csv\')\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n    for ax, mat, fmt, title in [(axes[0], cm, \'d\', \'Số mẫu\'), (axes[1], cm / np.maximum(cm.sum(1, keepdims=True), 1), \'.2f\', \'Chuẩn hóa theo nhãn thật\')]:\n        sns.heatmap(mat, annot=True, fmt=fmt, xticklabels=LABELS, yticklabels=LABELS, ax=ax, cmap=\'Blues\')\n        ax.set(xlabel=\'Dự đoán\', ylabel=\'Nhãn thật\', title=title)\n    fig.tight_layout(); fig.savefig(output / \'confusion_matrix.png\', dpi=160); plt.close(fig)\n    errors = predictions[predictions.label != predictions.prediction]\n    selected = []\n    for true in range(3):\n        for predicted in range(3):\n            if true != predicted:\n                candidates = errors[(errors.label == true) & (errors.prediction == predicted)]\n                selected.append(candidates.sample(min(10, len(candidates)), random_state=42))\n    pd.concat(selected).merge(frames[\'test\'][[\'sample_id\', \'text_raw\', \'text_segmented\']], on=\'sample_id\').to_csv(output / \'error_review.csv\', index=False)\n    history_path = output.parent / \'history.json\'\n    if history_path.exists():\n        history = json.loads(history_path.read_text())\n        fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n        axes[0].plot([r[\'epoch\'] for r in history], [r[\'train_loss\'] for r in history], label=\'train\')\n        axes[0].plot([r[\'epoch\'] for r in history], [r[\'validation\'][\'loss\'] for r in history], label=\'validation\')\n        axes[0].set(xlabel=\'Epoch\', ylabel=\'Loss\'); axes[0].legend()\n        axes[1].plot([r[\'epoch\'] for r in history], [r[\'validation\'][\'macro_f1\'] for r in history], marker=\'o\')\n        axes[1].set(xlabel=\'Epoch\', ylabel=\'Validation Macro-F1\')\n        fig.tight_layout(); fig.savefig(output / \'training_curves.png\', dpi=160); plt.close(fig)\n    print(\'FINAL_TEST_METRICS \' + json.dumps(result), flush=True)\n    return result\n'}
for relative_path, content in SOURCE_SNAPSHOT.items():
    target = ROOT / "src" / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
sys.path.insert(0, str(ROOT / "src"))
CONFIG = {'model_name': 'vinai/phobert-base', 'model_revision': 'main', 'seed': 42, 'max_length': 256, 'epochs': 5, 'learning_rate': 2e-05, 'train_batch_size': 8, 'eval_batch_size': 16, 'gradient_accumulation_steps': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.1, 'max_grad_norm': 1.0, 'save_steps': 500, 'run_name': 'phobert-clean-seed42'}
print("SOURCE_SHA256", '23c7e31a4283d4544ffeb958a79a5aa94e9258b46522d76064f4e84515536697')
(ROOT / "config.json").write_text(json.dumps(CONFIG, indent=2))


## 2. Audit, preprocessing và EDA

Các dòng trùng và xung đột nhãn được báo cáo, không tự deduplicate split chính thức. Các ví dụ kiểm tra preprocessing chỉ lấy từ train. Thống kê BPE được đo trước truncation, gồm special tokens.

In [ ]:
from vihsd.pipeline import prepare_all
STAGES = prepare_all(ROOT, CONFIG)
FRAMES, ENCODED, TOKENIZER, CONFIG = STAGES


In [ ]:
from IPython.display import display, Markdown, Image
import pandas as pd
EDA = ROOT / "reports/eda"
display(Markdown((EDA / "EDA_REPORT.md").read_text().split("![", 1)[0]))
display(pd.read_csv(EDA / "label_counts.csv"))
display(pd.read_csv(EDA / "ugc_rates.csv"))
for picture in sorted(EDA.glob("0*.png")):
    display(Image(filename=str(picture)))
print("Audit:")
print((ROOT / "reports/audit/audit.json").read_text())
print("Tokenization:")
print((EDA / "tokenization_summary.json").read_text())


## 3. Kiểm tra học và training đầy đủ

Smoke test dùng pretrained weights thật, kiểm tra loss/gradient, save/reload và optimizer state. Sau đó overfit 48 mẫu train. Model kiểm tra được hủy trước khi khởi tạo lại baseline với seed 42.

Full training chỉ dùng train và validation. Dự đoán test chạy một lần sau khi hoàn tất lựa chọn checkpoint. Checkpoint gần nhất (`last.pt`) chứa optimizer, scheduler và RNG để resume. Chỉ load file checkpoint do chính run này tạo.


In [ ]:
from vihsd.pipeline import run_training
RESULT = run_training(ROOT, CONFIG, stages=STAGES)


## 4. Kết quả và artifact

In [ ]:
RUN = ROOT / "runs" / CONFIG["run_name"]
display(Markdown((RUN / "BASELINE_REPORT.md").read_text()))
for name in ["training_curves.png", "confusion_matrix.png"]:
    display(Image(filename=str(RUN / "evaluation" / name)))
print("RUN_COMPLETE:", (ROOT / "RUN_COMPLETE.json").exists())
print("Best checkpoint:", RUN / "best")
print("Resume checkpoint:", RUN / "last.pt")
print("Predictions:", RUN / "evaluation/predictions.parquet")
